<a href="https://colab.research.google.com/github/RebootMe/Bookbot/blob/main/Searchable_encryption.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import secrets
import string
import os
import json

# --- Helpers ---
def normalize(word):
    """Lowercase and remove punctuation for matching."""
    return word.lower().strip(string.punctuation)

def encrypt_word(word, key_map):
    """Encrypt a normalized word using deterministic OTP-like encryption."""
    if word not in key_map:
        key = secrets.token_bytes(len(word.encode('utf-8')))
        key_map[word] = key
    else:
        key = key_map[word]

    word_bytes = word.encode('utf-8')
    encrypted_bytes = bytes(t ^ k for t, k in zip(word_bytes, key))
    return encrypted_bytes.hex()

def decrypt_word(enc_hex, key):
    enc_bytes = bytes.fromhex(enc_hex)
    dec_bytes = bytes(e ^ k for e, k in zip(enc_bytes, key))
    return dec_bytes.decode('utf-8')

def encrypt_paragraph(paragraph, key_map):
    """Encrypt paragraph word-by-word (normalized words)."""
    encrypted_words = []
    for word in paragraph.split():
        norm = normalize(word)
        enc_word = encrypt_word(norm, key_map)
        encrypted_words.append(enc_word)
    return encrypted_words

def decrypt_paragraph(enc_words, key_map):
    """Decrypt paragraph by matching encrypted words to their keys."""
    decrypted_words = []
    for enc_word in enc_words:
        for plain, key in key_map.items():
            if encrypt_word(plain, key_map) == enc_word:
                decrypted_words.append(plain)
                break
    return " ".join(decrypted_words)

def search_word(query, encrypted_data, key_map):
    """Search encrypted dataset for a query."""
    query_norm = normalize(query)
    enc_query = encrypt_word(query_norm, key_map)
    print(f"Encrypted query: {enc_query}")

    found = False
    for i, enc_p in enumerate(encrypted_data, 1):
        if enc_query in enc_p:
            found = True
            print(f"\nMatch in Paragraph {i} (Encrypted):", enc_p)
            print("Decrypted:", decrypt_paragraph(enc_p, key_map))

    if not found:
        print("No matches found.")

# --- Initial Data ---
paragraphs = [
    "The quick brown fox jumps over the lazy dog.",
    "Data science is an interdisciplinary field that uses scientific methods.",
    "Machine learning automates analytical model building.",
    "Natural Language Processing deals with human language.",
    "Artificial intelligence simulates human intelligence in machines."
]

encrypted_data = []
key_map = {}  # normalized_word -> key
data_dir = "paragraph_files"
os.makedirs(data_dir, exist_ok=True)

# --- File Functions ---
def save_plaintext_files():
    """Save initial plaintext paragraphs to files."""
    for i, para in enumerate(paragraphs, 1):
        with open(os.path.join(data_dir, f"paragraph_{i}.txt"), "w", encoding="utf-8") as f:
            f.write(para)
    print(f"Saved {len(paragraphs)} plaintext files in '{data_dir}'.")

def encrypt_files():
    """Encrypt paragraphs from files and overwrite them."""
    encrypted_data.clear()
    key_map.clear()
    for filename in sorted(os.listdir(data_dir)):
        if filename.startswith("paragraph_") and filename.endswith(".txt"):
            filepath = os.path.join(data_dir, filename)
            with open(filepath, "r", encoding="utf-8") as f:
                content = f.read()
            enc_p = encrypt_paragraph(content, key_map)
            encrypted_data.append(enc_p)
            with open(filepath, "w", encoding="utf-8") as f:
                f.write(" ".join(enc_p))  # save ciphertext to file
    # Save keys
    with open(os.path.join(data_dir, "keys.json"), "w", encoding="utf-8") as f:
        json.dump({k: v.hex() for k, v in key_map.items()}, f)
    print("Encryption complete. Files overwritten with ciphertext.")

def load_encrypted_data():
    """Load encrypted paragraphs and keys from files."""
    encrypted_data.clear()
    key_map.clear()
    # Load keys
    with open(os.path.join(data_dir, "keys.json"), "r", encoding="utf-8") as f:
        loaded_keys = json.load(f)
    key_map.update({k: bytes.fromhex(v) for k, v in loaded_keys.items()})
    # Load encrypted paragraphs
    for filename in sorted(os.listdir(data_dir)):
        if filename.startswith("paragraph_") and filename.endswith(".txt"):
            filepath = os.path.join(data_dir, filename)
            with open(filepath, "r", encoding="utf-8") as f:
                enc_words = f.read().split()
                encrypted_data.append(enc_words)

# --- Menu ---
def menu():
    while True:
        print("\n--- Searchable Encryption Menu ---")
        print("1. Save plaintext files")
        print("2. Encrypt and overwrite files")
        print("3. Search for a word")
        print("4. Exit")
        choice = input("Enter your choice: ")

        if choice == '1':
            save_plaintext_files()

        elif choice == '2':
            encrypt_files()

        elif choice == '3':
            if not encrypted_data:
                load_encrypted_data()
            query = input("Enter word to search: ")
            search_word(query, encrypted_data, key_map)

        elif choice == '4':
            break
        else:
            print("Invalid choice. Try again.")

menu()



--- Searchable Encryption Menu ---
1. Save plaintext files
2. Encrypt and overwrite files
3. Search for a word
4. Exit
Enter your choice: 1
Saved 5 plaintext files in 'paragraph_files'.

--- Searchable Encryption Menu ---
1. Save plaintext files
2. Encrypt and overwrite files
3. Search for a word
4. Exit
Enter your choice: 2
Encryption complete. Files overwritten with ciphertext.

--- Searchable Encryption Menu ---
1. Save plaintext files
2. Encrypt and overwrite files
3. Search for a word
4. Exit
Enter your choice: 3
Enter word to search: Data
Encrypted query: 56d3950b

Match in Paragraph 2 (Encrypted): ['56d3950b', 'cfed4ec34c01e2', '3f6b', '8ebc', '4ed22c816da4a254b62f1c0efdce1a4c8b', '505d43e08f', 'bf99efd5', '48ac483e', 'af07152d1983534f768a', '6b6d4197b97027']
Decrypted: data science is an interdisciplinary field that uses scientific methods

--- Searchable Encryption Menu ---
1. Save plaintext files
2. Encrypt and overwrite files
3. Search for a word
4. Exit
Enter your choice: 